## Basic Setup

In [2]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
tawsifurrahman_covid19_radiography_database_path = kagglehub.dataset_download('tawsifurrahman/covid19-radiography-database')

print('Data source import complete.')


Using Colab cache for faster access to the 'covid19-radiography-database' dataset.
Data source import complete.


In [3]:
!pip install grad-cam

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 4.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for grad-cam: filename=grad_cam-1.5.7-py3-none-any.whl size=53345 sha256=7faae21edbda2c3cea36cb53b61311ad083457e27af188b59ed79763935393be
  Stored in directory: /root/.cache/pip/wheels/91/36/f0/c2367865493d51b13369769181beca14603d17d6539a5922e3
Successfully built grad-cam


## Setup, Imports, and Configuration

In [4]:
!pip install grad-cam timm -q

import argparse
import copy
import json
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF
from tqdm import tqdm
import timm

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# We use a dummy class here to simulate argparse safely inside a notebook without crashing
class Args:
    data_dir = "/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset"
    output_dir = "./runs/vit_lung_attention"
    img_size = 224
    batch_size = 32
    seed = 42
    num_workers = 4
    phase1_epochs = 15
    phase1_lr = 1e-3
    phase2_epochs = 40
    phase2_lr = 1e-5
    unfreeze_blocks = 2
    optimizer = "adamw"
    weight_decay = 1e-4
    scheduler = "cosine"
    patience = 5

args = Args()
set_seed(args.seed)
Path(args.output_dir).mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


### Loss and training-loop functions (unchanged — re-run to refresh definitions)

In [5]:
def attention_guidance_loss(att_logits: torch.Tensor, masks: torch.Tensor, target_mode: str = "soft") -> torch.Tensor:
    if masks.ndim == 3:
        masks = masks.unsqueeze(1)
    masks = masks.float()
    h, w = att_logits.shape[-2:]
    target = F.adaptive_avg_pool2d(masks, (h, w)).clamp(0.0, 1.0)
    if target_mode == "hard":
        target = (target > 0.5).float()
    return F.binary_cross_entropy_with_logits(att_logits, target)

def compute_total_loss(class_logits, att_logits, labels, masks, classification_criterion, lambda_att: float = 0.5):
    cls_loss = classification_criterion(class_logits, labels)
    if att_logits is not None and lambda_att != 0.0:
        att_loss = attention_guidance_loss(att_logits, masks)
        total = cls_loss + lambda_att * att_loss
    else:
        att_loss = torch.zeros_like(cls_loss)
        total = cls_loss
    return total, cls_loss, att_loss

def run_epoch(model, loader, criterion, optimizer, device, train: bool, desc: str = ""):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    total_cls_loss, total_att_loss = 0.0, 0.0

    non_blocking = device.type == "cuda"
    context = torch.enable_grad() if train else torch.no_grad()
    progress = tqdm(loader, desc=desc, leave=False, dynamic_ncols=True)
    with context:
        for images, labels, masks in progress:
            images = images.to(device, non_blocking=non_blocking)
            labels = labels.to(device, non_blocking=non_blocking)
            masks = masks.to(device, non_blocking=non_blocking)

            if train:
                optimizer.zero_grad()

            outputs, att, att_logits = model(images)
            loss, cls_loss, att_loss = compute_total_loss(
                outputs, att_logits, labels, masks, criterion, lambda_att=0.5
            )

            if train:
                loss.backward()
                optimizer.step()

            batch_size = images.size(0)
            total_loss += loss.item() * batch_size
            total_cls_loss += cls_loss.item() * batch_size
            total_att_loss += att_loss.item() * batch_size

            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += batch_size
            progress.set_postfix(
                loss=f"{total_loss / total:.4f}",
                acc=f"{correct / total:.4f}",
                att_loss=f"{total_att_loss / total:.4f}"
            )

    return total_loss / total, correct / total

def train_phase(
    model, train_loader, val_loader, criterion, optimizer, scheduler,
    device, epochs, patience, phase_name, output_dir,
):
    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0
    history = []

    for epoch in range(1, epochs + 1):
        train_desc = f"{phase_name} epoch {epoch}/{epochs} train"
        val_desc = f"{phase_name} epoch {epoch}/{epochs} val"
        train_loss, train_acc = run_epoch(
            model, train_loader, criterion, optimizer, device, train=True, desc=train_desc
        )
        val_loss, val_acc = run_epoch(
            model, val_loader, criterion, optimizer, device, train=False, desc=val_desc
        )

        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_loss)
            else:
                scheduler.step()

        history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
                         "val_loss": val_loss, "val_acc": val_acc})
        print(f"[{phase_name}] epoch {epoch}/{epochs} "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"[{phase_name}] Early stopping at epoch {epoch} (no improvement for {patience} epochs).")
                break

    model.load_state_dict(best_state)
    with open(Path(output_dir) / f"{phase_name}_history.json", "w") as f:
        json.dump(history, f, indent=2)
    return model

@torch.no_grad()
def evaluate(model, loader, class_names, device, output_dir, evaluation_type):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []

    non_blocking = device.type == "cuda"
    for images, labels, masks in loader:
        images = images.to(device, non_blocking=non_blocking)
        outputs, att, att_logits = model(images)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)

        all_labels.extend(labels.numpy())
        all_preds.extend(preds)
        all_probs.extend(probs)

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)

    report = classification_report(all_labels, all_preds, target_names=class_names, digits=4, output_dict=True)
    cm = confusion_matrix(all_labels, all_preds)

    try:
        auc_macro = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro")
        auc_per_class = roc_auc_score(all_labels, all_probs, multi_class="ovr", average=None)
    except ValueError:
        auc_macro, auc_per_class = None, None

    results = {
        "classification_report": report,
        "confusion_matrix": cm.tolist(),
        "roc_auc_macro": auc_macro,
        "roc_auc_per_class": auc_per_class.tolist() if auc_per_class is not None else None,
        "class_names": class_names,
    }

    with open(Path(output_dir) / f"{evaluation_type}_test_results.json", "w") as f:
        json.dump(results, f, indent=2)

    print(f"\n=== Test set performance ({evaluation_type}) ===")
    print(f"Test accuracy: {accuracy_score(all_labels, all_preds):.4f}")
    print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))
    print("Confusion matrix:\n", cm)
    if auc_macro is not None:
        print(f"Macro ROC-AUC: {auc_macro:.4f}")

    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
    cm_df.to_csv(Path(output_dir) / f"{evaluation_type}_confusion_matrix.csv")

    summary = {
        "accuracy": accuracy_score(all_labels, all_preds),
        "macro_f1": report["macro avg"]["f1-score"],
        "macro_recall": report["macro avg"]["recall"],
        "macro_precision": report["macro avg"]["precision"],
        "weighted_f1": report["weighted avg"]["f1-score"],
        "weighted_recall": report["weighted avg"]["recall"],
        "roc_auc_macro": auc_macro,
    }
    pd.DataFrame([summary]).to_csv(Path(output_dir) / f"{evaluation_type}_summary_metrics.csv", index=False)
    return results

def build_optimizer(model, name: str, lr: float, weight_decay: float):
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    if name == "adamw":
        return torch.optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)
    elif name == "adam":
        return torch.optim.Adam(trainable_params, lr=lr, weight_decay=weight_decay)
    elif name == "sgd":
        return torch.optim.SGD(trainable_params, lr=lr, momentum=0.9, weight_decay=weight_decay)
    raise ValueError(f"Unknown optimizer: {name}")

def build_scheduler(optimizer, name: str, epochs: int):
    if name == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    elif name == "step":
        return torch.optim.lr_scheduler.StepLR(optimizer, step_size=max(1, epochs // 3), gamma=0.1)
    elif name == "plateau":
        return torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.1, patience=2)
    elif name == "none":
        return None
    raise ValueError(f"Unknown scheduler: {name}")

def freeze_vit_backbone(model: nn.Module):
    """Phase 1: freeze everything except the final classifier head and attention module."""
    for param in model.parameters():
        param.requires_grad = False

    for p in model.backbone.get_classifier().parameters():
        p.requires_grad = True

    if model.attn is not None:
        for p in model.attn.parameters():
            p.requires_grad = True

def unfreeze_vit_blocks(model: nn.Module, num_blocks_to_unfreeze: int = 2):
    """Phase 2: unfreeze the classifier, attention module, and the last N ViT blocks."""
    for param in model.parameters():
        param.requires_grad = False

    for p in model.backbone.get_classifier().parameters():
        p.requires_grad = True

    if model.attn is not None:
        for p in model.attn.parameters():
            p.requires_grad = True

    # Unfreeze final LayerNorms in ViT
    for name, param in model.backbone.named_parameters():
        if "norm" in name and "blocks" not in name:
            param.requires_grad = True

    # Unfreeze the requested number of target blocks (last to first)
    total_blocks = len(model.backbone.blocks)
    target_blocks = [f"blocks.{total_blocks - 1 - i}" for i in range(num_blocks_to_unfreeze)]

    for name, param in model.backbone.named_parameters():
        if any(b in name for b in target_blocks):
            param.requires_grad = True

In [6]:
import timm
import torch.nn as nn
import torch.nn.functional as F

class LungRegionAttention(nn.Module):
    def __init__(
        self, in_channels: int, reduction: int = 8, gate_mode: str = "residual", init_bias: float = 0.0,
    ) -> None:
        super().__init__()
        hidden = max(in_channels // reduction, 16)
        self.gate_mode = gate_mode
        self.conv1 = nn.Conv2d(in_channels, hidden, kernel_size=1, bias=True)
        self.act = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(hidden, 1, kernel_size=1, bias=True)

        nn.init.kaiming_normal_(self.conv1.weight, nonlinearity="relu")
        nn.init.zeros_(self.conv1.bias)
        nn.init.zeros_(self.conv2.weight)
        nn.init.constant_(self.conv2.bias, init_bias)

    def forward(self, feat: torch.Tensor):
        att_logits = self.conv2(self.act(self.conv1(feat)))
        att = torch.sigmoid(att_logits)
        if self.gate_mode == "residual":
            out = feat * (1.0 + att)
        elif self.gate_mode == "multiply":
            out = feat * att
        else:
            out = feat
        return out, att, att_logits

class ViTLungAttention(nn.Module):
    def __init__(
        self, num_classes: int = 4, pretrained: bool = True, use_attention: bool = True,
        reduction: int = 8, gate_mode: str = "residual", drop_rate: float = 0.0,
    ) -> None:
        super().__init__()
        # KEY FIX: global_pool='avg' — classifier reads the averaged patch tokens,
        # not just the CLS token. This is what makes the gate's output actually
        # reach the classifier.
        self.backbone = timm.create_model(
            'vit_base_patch16_224', pretrained=pretrained, num_classes=num_classes,
            global_pool='avg', drop_rate=drop_rate,
        )
        self.use_attention = use_attention
        c = self.backbone.num_features  # 768 for vit_base

        if not use_attention:
            self.attn = None
        else:
            self.attn = LungRegionAttention(c, reduction=reduction, gate_mode=gate_mode)

        self.post_attn = nn.Identity()  # Grad-CAM tap point

    def forward(self, x: torch.Tensor):
        f = self.backbone.forward_features(x)          # [B, 197, 768]

        # CLS token is discarded — never reaches the classifier under avg-pool.
        patch_tokens = f[:, 1:]                          # [B, 196, 768]
        B, N, C = patch_tokens.shape
        H = W = int(N ** 0.5)                             # 196 -> 14x14

        spatial_f = patch_tokens.transpose(1, 2).reshape(B, C, H, W)

        if self.attn is not None:
            spatial_f, att, att_logits = self.attn(spatial_f)
        else:
            att, att_logits = None, None

        spatial_f = self.post_attn(spatial_f)

        # Back to sequence, then average-pool the (possibly gated) patch tokens —
        # this pooled vector is what the classifier actually sees now.
        gated_patches = spatial_f.flatten(2).transpose(1, 2)   # [B, 196, 768]
        pooled = gated_patches.mean(dim=1)                      # [B, 768]

        # Manually replicate timm's forward_head logic for global_pool='avg'
        pooled = self.backbone.fc_norm(pooled)
        pooled = self.backbone.head_drop(pooled)
        logits = self.backbone.head(pooled)

        return logits, att, att_logits

# Test the model with a dummy batch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_classes = 4

model = ViTLungAttention(
    num_classes=num_classes,
    pretrained=True
).to(device)

# Dummy images: [batch, channels, height, width]
imgs = torch.randn(2, 3, 224, 224).to(device)

out, att, att_logits = model(imgs)

print(f"Output shape: {out.shape}")
print(f"Attention shape: {att.shape}")
print(f"Attention logits shape: {att_logits.shape}")

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Output shape: torch.Size([2, 4])
Attention shape: torch.Size([2, 1, 14, 14])
Attention logits shape: torch.Size([2, 1, 14, 14])


## Dataset and Dataloaders (with masks)

In [7]:
data_dir = "/kaggle/input/covid19-radiography-database/COVID-19_Radiography_Dataset"

print(Path(data_dir).exists())
print(Path(data_dir))

True
/kaggle/input/covid19-radiography-database/COVID-19_Radiography_Dataset


## The ViT Lung Attention Model

In [8]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def stratified_split(dataset: ImageFolder, seed: int = 42):
    targets = np.array(dataset.targets)
    indices = np.arange(len(dataset))
    train_idx, temp_idx = train_test_split(indices, test_size=0.3, stratify=targets, random_state=seed)
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, stratify=targets[temp_idx], random_state=seed)
    return train_idx, val_idx, test_idx

class JointTransform:
    def __init__(self, img_size: int = 224, train: bool = True):
        self.img_size = img_size
        self.train = train
        self.color_jitter = transforms.ColorJitter(brightness=0.2, contrast=0.2)

    def __call__(self, image: Image.Image, mask: Image.Image):
        image = TF.resize(image, [self.img_size, self.img_size], interpolation=InterpolationMode.BILINEAR)
        mask = TF.resize(mask, [self.img_size, self.img_size], interpolation=InterpolationMode.NEAREST)

        if self.train:
            image = TF.pad(image, padding=8, padding_mode="reflect")
            mask = TF.pad(mask, padding=8, fill=0)
            i, j, h, w = transforms.RandomCrop.get_params(image, output_size=(self.img_size, self.img_size))
            image = TF.crop(image, i, j, h, w)
            mask = TF.crop(mask, i, j, h, w)
            if random.random() < 0.5:
                image = TF.hflip(image)
                mask = TF.hflip(mask)
            angle = random.uniform(-10, 10)
            image = TF.rotate(image, angle, interpolation=InterpolationMode.BILINEAR)
            mask = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST)
            image = self.color_jitter(image)

        image = TF.to_grayscale(image, num_output_channels=3)
        image = TF.to_tensor(image)
        image = TF.normalize(image, mean=IMAGENET_MEAN, std=IMAGENET_STD)

        mask = TF.to_grayscale(mask, num_output_channels=1)
        mask = TF.to_tensor(mask)
        mask = (mask > 0.5).float()
        return image, mask

class CXRWithMaskDataset(Dataset):
    def __init__(self, base_dataset, indices, transform):
        self.base_dataset = base_dataset
        self.indices = indices
        self.transform = transform

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, idx: int):
        base_idx = self.indices[idx]
        image_path, label = self.base_dataset.samples[base_idx]
        image_path = Path(image_path)
        mask_path = image_path.parent.parent / "masks" / image_path.name

        if not mask_path.exists():
            raise FileNotFoundError(f"Mask not found for: {image_path}\\nExpected mask: {mask_path}")

        image = Image.open(image_path).convert("L")
        mask = Image.open(mask_path).convert("L")
        image, mask = self.transform(image, mask)
        return image, label, mask

def compute_class_weights(train_targets: np.ndarray, num_classes: int) -> torch.Tensor:
    counts = np.bincount(train_targets, minlength=num_classes).astype(np.float32)
    counts[counts == 0] = 1.0
    weights = counts.sum() / (num_classes * counts)
    return torch.tensor(weights, dtype=torch.float32)

def build_dataloaders(data_dir: str, img_size: int, batch_size: int, seed: int, num_workers: int = 4):
    def only_images_folder(path):
        p = Path(path)
        return p.parent.name.lower() == "images" and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".tiff"}

    base_dataset = ImageFolder(root=data_dir, is_valid_file=only_images_folder)
    train_idx, val_idx, test_idx = stratified_split(base_dataset, seed=seed)

    train_tf = JointTransform(img_size=img_size, train=True)
    eval_tf = JointTransform(img_size=img_size, train=False)

    train_ds = CXRWithMaskDataset(base_dataset, train_idx, train_tf)
    val_ds = CXRWithMaskDataset(base_dataset, val_idx, eval_tf)
    test_ds = CXRWithMaskDataset(base_dataset, test_idx, eval_tf)

    pin_memory = torch.cuda.is_available()
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)

    class_names = base_dataset.classes
    train_targets = np.array(base_dataset.targets)[train_idx]
    return train_loader, val_loader, test_loader, class_names, train_targets

data_dir = "/kaggle/input/covid19-radiography-database/COVID-19_Radiography_Dataset"

train_loader, val_loader, test_loader, class_names, train_targets = build_dataloaders(
    data_dir,
    args.img_size,
    args.batch_size,
    args.seed,
    args.num_workers
)

num_classes = len(class_names)

print(f"Classes ({num_classes}): {class_names}")

class_weights = compute_class_weights(
    train_targets,
    num_classes
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

Classes (4): ['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


## T48 — Validate ViT-avgpool Baseline (A0)

Per T26 spec 6: the ViT-avgpool classifier is architecturally different from T17's CLS-token baseline
(91.91% accuracy), so it must be trained and sanity-checked on its own *before* it can serve as the
control for any attention arm (A1–A6). This section trains A0 with `use_attention=False` — same data,
same seed, same hyperparameters as the ablation run — to get a valid reference accuracy.

**Acceptance check (13, criterion A0'):** test accuracy should land in a plausible range vs. T17's
91.91%. A collapse to ~70% would mean the avg-pool wiring is broken, not that avg-pooling is worse.

In [13]:
args.output_dir = "./runs/vit_lung_attention_A0_baseline"
Path(args.output_dir).mkdir(parents=True, exist_ok=True)
print(f"Output directory for this run: {args.output_dir}")

Output directory for this run: ./runs/vit_lung_attention_A0_baseline


**bold text** Build the A0 model (attention module disabled)

In [14]:
import timm
import torch.nn as nn
import torch.nn.functional as F

class LungRegionAttention(nn.Module):
    def __init__(
        self, in_channels: int, reduction: int = 8, gate_mode: str = "residual", init_bias: float = 0.0,
    ) -> None:
        super().__init__()
        hidden = max(in_channels // reduction, 16)
        self.gate_mode = gate_mode
        self.conv1 = nn.Conv2d(in_channels, hidden, kernel_size=1, bias=True)
        self.act = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(hidden, 1, kernel_size=1, bias=True)

        nn.init.kaiming_normal_(self.conv1.weight, nonlinearity="relu")
        nn.init.zeros_(self.conv1.bias)
        nn.init.zeros_(self.conv2.weight)
        nn.init.constant_(self.conv2.bias, init_bias)

    def forward(self, feat: torch.Tensor):
        att_logits = self.conv2(self.act(self.conv1(feat)))
        att = torch.sigmoid(att_logits)
        if self.gate_mode == "residual":
            out = feat * (1.0 + att)
        elif self.gate_mode == "multiply":
            out = feat * att
        else:
            out = feat
        return out, att, att_logits

class ViTLungAttention(nn.Module):
    def __init__(
        self, num_classes: int = 4, pretrained: bool = True, use_attention: bool = True,
        reduction: int = 8, gate_mode: str = "residual", drop_rate: float = 0.0,
    ) -> None:
        super().__init__()
        self.backbone = timm.create_model(
            'vit_base_patch16_224', pretrained=pretrained, num_classes=num_classes,
            global_pool='avg', drop_rate=drop_rate,
        )
        self.use_attention = use_attention
        c = self.backbone.num_features  # 768 for vit_base

        if not use_attention:
            self.attn = None
        else:
            self.attn = LungRegionAttention(c, reduction=reduction, gate_mode=gate_mode)

        self.post_attn = nn.Identity()  # Grad-CAM tap point

    def forward(self, x: torch.Tensor):
        f = self.backbone.forward_features(x)          # [B, 197, 768]

        patch_tokens = f[:, 1:]                          # [B, 196, 768]
        B, N, C = patch_tokens.shape
        H = W = int(N ** 0.5)                             # 196 -> 14x14

        spatial_f = patch_tokens.transpose(1, 2).reshape(B, C, H, W)

        if self.attn is not None:
            spatial_f, att, att_logits = self.attn(spatial_f)
        else:
            att, att_logits = None, None

        spatial_f = self.post_attn(spatial_f)

        gated_patches = spatial_f.flatten(2).transpose(1, 2)   # [B, 196, 768]
        pooled = gated_patches.mean(dim=1)                      # [B, 768]

        pooled = self.backbone.fc_norm(pooled)
        pooled = self.backbone.head_drop(pooled)
        logits = self.backbone.head(pooled)

        return logits, att, att_logits

# Test the model on a dummy batch to ensure shapes map correctly
imgs, labels, masks = next(iter(train_loader))
model = ViTLungAttention(num_classes=num_classes, use_attention=False).to(device)
out, att, att_logits = model(imgs.to(device))
print(f"Output shape: {out.shape}")
print(f"Attention shape: {att}")   # expected: None, since use_attention=False

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Output shape: torch.Size([32, 4])
Attention shape: None


## λ_att Sweep

Before training the full attention arms (A1–A5), sweep `λ_att` over a short schedule to pick a
reasonable value, rather than assuming one. Each trial here trains Phase 1 only (frozen backbone,
few epochs) with the gate active (`use_attention=True`, `gate_mode="residual"`) — enough to compare
relative trends in validation accuracy and attention loss without spending a full Phase 1+2 budget
per value. The winning λ_att carries forward into the real A1–A5 training runs.

In [ ]:
def attention_guidance_loss(att_logits: torch.Tensor, masks: torch.Tensor, target_mode: str = "soft") -> torch.Tensor:
    if masks.ndim == 3:
        masks = masks.unsqueeze(1)
    masks = masks.float()
    h, w = att_logits.shape[-2:]
    target = F.adaptive_avg_pool2d(masks, (h, w)).clamp(0.0, 1.0)
    if target_mode == "hard":
        target = (target > 0.5).float()
    return F.binary_cross_entropy_with_logits(att_logits, target)

def compute_total_loss(class_logits, att_logits, labels, masks, classification_criterion, lambda_att: float = 0.5):
    cls_loss = classification_criterion(class_logits, labels)
    if att_logits is not None and lambda_att != 0.0:
        att_loss = attention_guidance_loss(att_logits, masks)
        total = cls_loss + lambda_att * att_loss
    else:
        att_loss = torch.zeros_like(cls_loss)
        total = cls_loss
    return total, cls_loss, att_loss

def run_epoch(model, loader, criterion, optimizer, device, train: bool, desc: str = "", lambda_att: float = 0.5):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    total_cls_loss, total_att_loss = 0.0, 0.0

    non_blocking = device.type == "cuda"
    context = torch.enable_grad() if train else torch.no_grad()
    progress = tqdm(loader, desc=desc, leave=False, dynamic_ncols=True)
    with context:
        for images, labels, masks in progress:
            images = images.to(device, non_blocking=non_blocking)
            labels = labels.to(device, non_blocking=non_blocking)
            masks = masks.to(device, non_blocking=non_blocking)

            if train:
                optimizer.zero_grad()

            outputs, att, att_logits = model(images)
            loss, cls_loss, att_loss = compute_total_loss(
                outputs, att_logits, labels, masks, criterion, lambda_att=lambda_att
            )

            if train:
                loss.backward()
                optimizer.step()

            batch_size = images.size(0)
            total_loss += loss.item() * batch_size
            total_cls_loss += cls_loss.item() * batch_size
            total_att_loss += att_loss.item() * batch_size

            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += batch_size
            progress.set_postfix(
                loss=f"{total_loss / total:.4f}",
                acc=f"{correct / total:.4f}",
                att_loss=f"{total_att_loss / total:.4f}"
            )

    return total_loss / total, correct / total

def train_phase(
    model, train_loader, val_loader, criterion, optimizer, scheduler,
    device, epochs, patience, phase_name, output_dir, lambda_att: float = 0.5,
):
    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0
    history = []

    for epoch in range(1, epochs + 1):
        train_desc = f"{phase_name} epoch {epoch}/{epochs} train"
        val_desc = f"{phase_name} epoch {epoch}/{epochs} val"
        train_loss, train_acc = run_epoch(
            model, train_loader, criterion, optimizer, device, train=True, desc=train_desc, lambda_att=lambda_att
        )
        val_loss, val_acc = run_epoch(
            model, val_loader, criterion, optimizer, device, train=False, desc=val_desc, lambda_att=lambda_att
        )

        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_loss)
            else:
                scheduler.step()

        history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
                         "val_loss": val_loss, "val_acc": val_acc})
        print(f"[{phase_name}] epoch {epoch}/{epochs} "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"[{phase_name}] Early stopping at epoch {epoch} (no improvement for {patience} epochs).")
                break

    model.load_state_dict(best_state)
    with open(Path(output_dir) / f"{phase_name}_history.json", "w") as f:
        json.dump(history, f, indent=2)
    return model

@torch.no_grad()
def evaluate(model, loader, class_names, device, output_dir, evaluation_type):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []

    non_blocking = device.type == "cuda"
    for images, labels, masks in loader:
        images = images.to(device, non_blocking=non_blocking)
        outputs, att, att_logits = model(images)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)

        all_labels.extend(labels.numpy())
        all_preds.extend(preds)
        all_probs.extend(probs)

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)

    report = classification_report(all_labels, all_preds, target_names=class_names, digits=4, output_dict=True)
    cm = confusion_matrix(all_labels, all_preds)

    try:
        auc_macro = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro")
        auc_per_class = roc_auc_score(all_labels, all_probs, multi_class="ovr", average=None)
    except ValueError:
        auc_macro, auc_per_class = None, None

    results = {
        "classification_report": report,
        "confusion_matrix": cm.tolist(),
        "roc_auc_macro": auc_macro,
        "roc_auc_per_class": auc_per_class.tolist() if auc_per_class is not None else None,
        "class_names": class_names,
    }

    with open(Path(output_dir) / f"{evaluation_type}_test_results.json", "w") as f:
        json.dump(results, f, indent=2)

    print(f"\n=== Test set performance ({evaluation_type}) ===")
    print(f"Test accuracy: {accuracy_score(all_labels, all_preds):.4f}")
    print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))
    print("Confusion matrix:\n", cm)
    if auc_macro is not None:
        print(f"Macro ROC-AUC: {auc_macro:.4f}")

    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
    cm_df.to_csv(Path(output_dir) / f"{evaluation_type}_confusion_matrix.csv")

    summary = {
        "accuracy": accuracy_score(all_labels, all_preds),
        "macro_f1": report["macro avg"]["f1-score"],
        "macro_recall": report["macro avg"]["recall"],
        "macro_precision": report["macro avg"]["precision"],
        "weighted_f1": report["weighted avg"]["f1-score"],
        "weighted_recall": report["weighted avg"]["recall"],
        "roc_auc_macro": auc_macro,
    }
    pd.DataFrame([summary]).to_csv(Path(output_dir) / f"{evaluation_type}_summary_metrics.csv", index=False)
    return results

def build_optimizer(model, name: str, lr: float, weight_decay: float):
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    if name == "adamw":
        return torch.optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)
    elif name == "adam":
        return torch.optim.Adam(trainable_params, lr=lr, weight_decay=weight_decay)
    elif name == "sgd":
        return torch.optim.SGD(trainable_params, lr=lr, momentum=0.9, weight_decay=weight_decay)
    raise ValueError(f"Unknown optimizer: {name}")

def build_scheduler(optimizer, name: str, epochs: int):
    if name == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    elif name == "step":
        return torch.optim.lr_scheduler.StepLR(optimizer, step_size=max(1, epochs // 3), gamma=0.1)
    elif name == "plateau":
        return torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.1, patience=2)
    elif name == "none":
        return None
    raise ValueError(f"Unknown scheduler: {name}")

def freeze_vit_backbone(model: nn.Module):
    """Phase 1: freeze everything except the final classifier head and attention module."""
    for param in model.parameters():
        param.requires_grad = False

    for p in model.backbone.get_classifier().parameters():
        p.requires_grad = True

    if model.attn is not None:
        for p in model.attn.parameters():
            p.requires_grad = True

def unfreeze_vit_blocks(model: nn.Module, num_blocks_to_unfreeze: int = 2):
    """Phase 2: unfreeze the classifier, attention module, and the last N ViT blocks."""
    for param in model.parameters():
        param.requires_grad = False

    for p in model.backbone.get_classifier().parameters():
        p.requires_grad = True

    if model.attn is not None:
        for p in model.attn.parameters():
            p.requires_grad = True

    # Unfreeze final LayerNorms in ViT
    for name, param in model.backbone.named_parameters():
        if "norm" in name and "blocks" not in name:
            param.requires_grad = True

    # Unfreeze the requested number of target blocks (last to first)
    total_blocks = len(model.backbone.blocks)
    target_blocks = [f"blocks.{total_blocks - 1 - i}" for i in range(num_blocks_to_unfreeze)]

    for name, param in model.backbone.named_parameters():
        if any(b in name for b in target_blocks):
            param.requires_grad = True

In [ ]:
sweep_lambda_values = [0.1, 0.3, 0.5, 1.0]  # 0.0 already covered by A0
sweep_epochs = 3
sweep_results = []

for lam in sweep_lambda_values:
    print(f"\n=== Sweep trial: lambda_att={lam} ===")
    sweep_model = ViTLungAttention(
        num_classes=num_classes, use_attention=True, gate_mode="residual"
    ).to(device)
    freeze_vit_backbone(sweep_model)
    sweep_opt = build_optimizer(sweep_model, args.optimizer, args.phase1_lr, args.weight_decay)
    sweep_sched = build_scheduler(sweep_opt, args.scheduler, sweep_epochs)

    sweep_out_dir = f"./runs/lambda_sweep/lambda_{lam}"
    Path(sweep_out_dir).mkdir(parents=True, exist_ok=True)

    sweep_model = train_phase(
        sweep_model, train_loader, val_loader, criterion, sweep_opt, sweep_sched,
        device, sweep_epochs, patience=sweep_epochs, phase_name=f"sweep_lambda_{lam}",
        output_dir=sweep_out_dir, lambda_att=lam,
    )

    val_loss, val_acc = run_epoch(
        sweep_model, val_loader, criterion, None, device, train=False,
        desc=f"final_val_lambda_{lam}", lambda_att=lam,
    )
    sweep_results.append({"lambda_att": lam, "val_loss": val_loss, "val_acc": val_acc})

sweep_df = pd.DataFrame(sweep_results)
print("\n=== Sweep summary ===")
print(sweep_df)
sweep_df.to_csv("./runs/lambda_sweep/sweep_summary.csv", index=False)


=== Sweep trial: lambda_att=0.1 ===


Unexpected keys (norm.bias, norm.weight) found while loading pretrained weights. This may be expected if model is being adapted.


[sweep_lambda_0.1] epoch 1/3 train_loss=0.6905 train_acc=0.7201 val_loss=0.5701 val_acc=0.7575


[sweep_lambda_0.1] epoch 2/3 train_loss=0.4956 train_acc=0.7980 val_loss=0.4656 val_acc=0.8252


[sweep_lambda_0.1] epoch 3/3 train_loss=0.4498 train_acc=0.8155 val_loss=0.4470 val_acc=0.8170


...

Unexpected keys (norm.bias, norm.weight) found while loading pretrained weights. This may be expected if model is being adapted.


[sweep_lambda_1.0] epoch 1/3 train_loss=0.8820 train_acc=0.7099 val_loss=0.7026 val_acc=0.7792


[sweep_lambda_1.0] epoch 2/3 train_loss=0.6438 train_acc=0.7983 val_loss=0.6292 val_acc=0.8028


[sweep_lambda_1.0] epoch 3/3 train_loss=0.5917 train_acc=0.8213 val_loss=0.5874 val_acc=0.8302



=== Sweep summary ===
   lambda_att  val_loss   val_acc
0         0.1  0.446984  0.817008
1         0.3  0.482951  0.841260
2         0.5  0.512623  0.821732
3         1.0  0.587439  0.830236


### λ_att sweep result

Swept λ_att ∈ {0.1, 0.3, 0.5, 1.0}, 3 epochs each, frozen backbone only, single run per value.

| λ_att | val_loss | val_acc |
|-------|----------|---------|
| 0.1   | 0.4470   | 0.8170  |
| 0.3   | 0.4830   | 0.8413  |
| 0.5   | 0.5126   | 0.8217  |
| 1.0   | 0.5874   | 0.8302  |

Note: val_loss is not directly comparable across λ values, since it scales with λ (`total_loss =
cls_loss + λ_att * att_loss`). val_acc is the fair comparison metric.

**Chosen: λ_att = 0.3** (highest val_acc). Carried forward into A1–A5 training.

## Six-Arm Ablation Training — A0, A1, A2, A3, A4, A5 (seed 42, λ_att=0.3)

Per 14 step 5: train all six arms under identical conditions (same seed, full Phase 1 + Phase 2
schedule) so results are directly comparable. λ_att=0.3 (from the sweep above) is used for every
mask-supervised arm; A1 and A5 use λ_att=0 by design (§8's table).

| Arm | attention | gate_mode | λ_att |
|-----|-----------|-----------|-------|
| A0 vanilla | none | — | 0 |
| A1 gate-only | lung | residual | 0 |
| A2 full | lung | residual | 0.3 |
| A3 multiply-gate | lung | multiply | 0.3 |
| A4 guidance-only | lung | none | 0.3 |
| A5 CBAM | cbam | — | 0 |

## Adds CBAM + an attention selector

In [9]:
import timm
import torch.nn as nn
import torch.nn.functional as F

class LungRegionAttention(nn.Module):
    def __init__(
        self, in_channels: int, reduction: int = 8, gate_mode: str = "residual", init_bias: float = 0.0,
    ) -> None:
        super().__init__()
        hidden = max(in_channels // reduction, 16)
        self.gate_mode = gate_mode
        self.conv1 = nn.Conv2d(in_channels, hidden, kernel_size=1, bias=True)
        self.act = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(hidden, 1, kernel_size=1, bias=True)

        nn.init.kaiming_normal_(self.conv1.weight, nonlinearity="relu")
        nn.init.zeros_(self.conv1.bias)
        nn.init.zeros_(self.conv2.weight)
        nn.init.constant_(self.conv2.bias, init_bias)

    def forward(self, feat: torch.Tensor):
        att_logits = self.conv2(self.act(self.conv1(feat)))
        att = torch.sigmoid(att_logits)
        if self.gate_mode == "residual":
            out = feat * (1.0 + att)
        elif self.gate_mode == "multiply":
            out = feat * att
        else:
            out = feat  # A4: supervise only, don't gate
        return out, att, att_logits

class CBAMSpatialAttention(nn.Module):
    """CBAM spatial branch (Woo et al. 2018) -- comparator for arm A5.
    Multiplicative gate, no logits (A5 is never mask-supervised)."""
    def __init__(self, kernel_size: int = 7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)

    def forward(self, feat: torch.Tensor):
        avg_out = feat.mean(dim=1, keepdim=True)
        max_out = feat.amax(dim=1, keepdim=True)
        att = torch.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))
        return feat * att, att, None

class ViTLungAttention(nn.Module):
    def __init__(
        self, num_classes: int = 4, pretrained: bool = True, use_attention: bool = True,
        reduction: int = 8, gate_mode: str = "residual", drop_rate: float = 0.0,
        attention: str = "lung",  # "lung" | "cbam"
    ) -> None:
        super().__init__()
        self.backbone = timm.create_model(
            'vit_base_patch16_224', pretrained=pretrained, num_classes=num_classes,
            global_pool='avg', drop_rate=drop_rate,
        )
        self.use_attention = use_attention
        self.attention_kind = attention
        c = self.backbone.num_features  # 768 for vit_base

        if not use_attention:
            self.attn = None
        elif attention == "lung":
            self.attn = LungRegionAttention(c, reduction=reduction, gate_mode=gate_mode)
        elif attention == "cbam":
            self.attn = CBAMSpatialAttention()
        else:
            raise ValueError(f"unknown attention: {attention}")

        self.post_attn = nn.Identity()  # Grad-CAM tap point

    def forward(self, x: torch.Tensor):
        f = self.backbone.forward_features(x)          # [B, 197, 768]

        patch_tokens = f[:, 1:]                          # [B, 196, 768]
        B, N, C = patch_tokens.shape
        H = W = int(N ** 0.5)                             # 196 -> 14x14

        spatial_f = patch_tokens.transpose(1, 2).reshape(B, C, H, W)

        if self.attn is not None:
            spatial_f, att, att_logits = self.attn(spatial_f)
        else:
            att, att_logits = None, None

        spatial_f = self.post_attn(spatial_f)

        gated_patches = spatial_f.flatten(2).transpose(1, 2)   # [B, 196, 768]
        pooled = gated_patches.mean(dim=1)                      # [B, 768]

        pooled = self.backbone.fc_norm(pooled)
        pooled = self.backbone.head_drop(pooled)
        logits = self.backbone.head(pooled)

        return logits, att, att_logits

# Sanity check
imgs, labels, masks = next(iter(train_loader))
model = ViTLungAttention(num_classes=num_classes, use_attention=False).to(device)
out, att, att_logits = model(imgs.to(device))
print(f"Output shape: {out.shape}")
print(f"Attention: {att}")

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Output shape: torch.Size([32, 4])
Attention: None


## The six-arm loop

In [ ]:
LAMBDA_STAR = 0.3  # from the sweep

ARM_CONFIGS = {
    "A0_vanilla":       dict(use_attention=False, attention="lung", gate_mode="residual", lambda_att=0.0),
    "A1_gate_only":     dict(use_attention=True,  attention="lung", gate_mode="residual", lambda_att=0.0),
    "A2_full":          dict(use_attention=True,  attention="lung", gate_mode="residual", lambda_att=LAMBDA_STAR),
    "A3_multiply_gate": dict(use_attention=True,  attention="lung", gate_mode="multiply", lambda_att=LAMBDA_STAR),
    "A4_guidance_only": dict(use_attention=True,  attention="lung", gate_mode="none",     lambda_att=LAMBDA_STAR),
    "A5_cbam":          dict(use_attention=True,  attention="cbam", gate_mode="residual", lambda_att=0.0),
}

arm_results = {}

for arm_name, cfg in ARM_CONFIGS.items():
    print(f"\n{'='*20} ARM: {arm_name} {'='*20}")
    set_seed(args.seed)  # re-seed before every arm, per spec

    arm_model = ViTLungAttention(
        num_classes=num_classes,
        use_attention=cfg["use_attention"],
        attention=cfg["attention"],
        gate_mode=cfg["gate_mode"],
    ).to(device)

    arm_out_dir = f"./runs/six_arm/{arm_name}"
    Path(arm_out_dir).mkdir(parents=True, exist_ok=True)

    # Phase 1
    freeze_vit_backbone(arm_model)
    opt1 = build_optimizer(arm_model, args.optimizer, args.phase1_lr, args.weight_decay)
    sched1 = build_scheduler(opt1, args.scheduler, args.phase1_epochs)
    arm_model = train_phase(
        arm_model, train_loader, val_loader, criterion, opt1, sched1,
        device, args.phase1_epochs, args.patience, f"{arm_name}_phase1",
        arm_out_dir, lambda_att=cfg["lambda_att"],
    )
    evaluate(arm_model, test_loader, class_names, device, arm_out_dir, "frozen")

    # Phase 2
    unfreeze_vit_blocks(arm_model, num_blocks_to_unfreeze=args.unfreeze_blocks)
    opt2 = build_optimizer(arm_model, args.optimizer, args.phase2_lr, args.weight_decay)
    sched2 = build_scheduler(opt2, args.scheduler, args.phase2_epochs)
    arm_model = train_phase(
        arm_model, train_loader, val_loader, criterion, opt2, sched2,
        device, args.phase2_epochs, args.patience, f"{arm_name}_phase2",
        arm_out_dir, lambda_att=cfg["lambda_att"],
    )
    result = evaluate(arm_model, test_loader, class_names, device, arm_out_dir, "finetuned")

    arm_results[arm_name] = {
        "accuracy": result["classification_report"]["accuracy"],
        "macro_f1": result["classification_report"]["macro avg"]["f1-score"],
    }

    torch.save(arm_model.state_dict(), Path(arm_out_dir) / "final_model.pt")

print("\n=== Six-arm summary ===")
arm_summary_df = pd.DataFrame(arm_results).T
print(arm_summary_df)
arm_summary_df.to_csv("./runs/six_arm/six_arm_summary.csv")


==================== ARM: A0_vanilla ====================


Unexpected keys (norm.bias, norm.weight) found while loading pretrained weights. This may be expected if model is being adapted.


[A0_vanilla_phase1] epoch 1/15 train_loss=0.6808 train_acc=0.7124 val_loss=0.5268 val_acc=0.7846


[A0_vanilla_phase1] epoch 2/15 train_loss=0.4714 train_acc=0.7951 val_loss=0.4343 val_acc=0.8076


[A0_vanilla_phase1] epoch 3/15 train_loss=0.4216 train_acc=0.8165 val_loss=0.4056 val_acc=0.8463


...

[A4_guidance_only_phase2] epoch 13/40 train_loss=0.1962 train_acc=0.9262 val_loss=0.3094 val_acc=0.9241


[A4_guidance_only_phase2] epoch 14/40 train_loss=0.1824 train_acc=0.9312 val_loss=0.2201 val_acc=0.9250


[A4_guidance_only_phase2] epoch 15/40 train_loss=0.1818 train_acc=0.9315 val_loss=0.2599 val_acc=0.9200


[A4_guidance_only_phase2] epoch 16/40 train_loss=0.1803 train_acc=0.9323 val_loss=0.2703 val_acc=0.9244


A4_guidance_only_phase2 epoch 17/40 train:  66%|██████▌   | 305/463 [02:39<01:21,  1.93it/s, acc=0.9357, att_loss=0.1478, loss=0.1687]

## A4 and A5

In [ ]:
LAMBDA_STAR = 0.3  # from the sweep

ARM_CONFIGS = {
    "A4_guidance_only": dict(use_attention=True,  attention="lung", gate_mode="none",     lambda_att=LAMBDA_STAR),
    "A5_cbam":          dict(use_attention=True,  attention="cbam", gate_mode="residual", lambda_att=0.0),
}

arm_results = {}

for arm_name, cfg in ARM_CONFIGS.items():
    print(f"\n{'='*20} ARM: {arm_name} {'='*20}")
    set_seed(args.seed)  # re-seed before every arm, per spec

    arm_model = ViTLungAttention(
        num_classes=num_classes,
        use_attention=cfg["use_attention"],
        attention=cfg["attention"],
        gate_mode=cfg["gate_mode"],
    ).to(device)

    arm_out_dir = f"./runs/six_arm/{arm_name}"
    Path(arm_out_dir).mkdir(parents=True, exist_ok=True)

    # Phase 1
    freeze_vit_backbone(arm_model)
    opt1 = build_optimizer(arm_model, args.optimizer, args.phase1_lr, args.weight_decay)
    sched1 = build_scheduler(opt1, args.scheduler, args.phase1_epochs)
    arm_model = train_phase(
        arm_model, train_loader, val_loader, criterion, opt1, sched1,
        device, args.phase1_epochs, args.patience, f"{arm_name}_phase1",
        arm_out_dir, lambda_att=cfg["lambda_att"],
    )
    evaluate(arm_model, test_loader, class_names, device, arm_out_dir, "frozen")

    # Phase 2
    unfreeze_vit_blocks(arm_model, num_blocks_to_unfreeze=args.unfreeze_blocks)
    opt2 = build_optimizer(arm_model, args.optimizer, args.phase2_lr, args.weight_decay)
    sched2 = build_scheduler(opt2, args.scheduler, args.phase2_epochs)
    arm_model = train_phase(
        arm_model, train_loader, val_loader, criterion, opt2, sched2,
        device, args.phase2_epochs, args.patience, f"{arm_name}_phase2",
        arm_out_dir, lambda_att=cfg["lambda_att"],
    )
    result = evaluate(arm_model, test_loader, class_names, device, arm_out_dir, "finetuned")

    arm_results[arm_name] = {
        "accuracy": result["classification_report"]["accuracy"],
        "macro_f1": result["classification_report"]["macro avg"]["f1-score"],
    }

    torch.save(arm_model.state_dict(), Path(arm_out_dir) / "final_model.pt")

print("\n=== Six-arm summary ===")
arm_summary_df = pd.DataFrame(arm_results).T
print(arm_summary_df)
arm_summary_df.to_csv("./runs/six_arm/six_arm_summary.csv")


==================== ARM: A4_guidance_only ====================


[A4_guidance_only_phase1] epoch 1/15 train_loss=0.7487 train_acc=0.7081 val_loss=0.5720 val_acc=0.7729


[A4_guidance_only_phase1] epoch 2/15 train_loss=0.5248 train_acc=0.7935 val_loss=0.4834 val_acc=0.8167


[A4_guidance_only_phase1] epoch 3/15 train_loss=0.4699 train_acc=0.8194 val_loss=0.4493 val_acc=0.8265


[A4_guidance_only_phase1] epoch 4/15 train_loss=0.4377 train_acc=0.8308 val_loss=0.4174 val_acc=0.8328


...

[A5_cbam_phase2] epoch 20/40 train_loss=0.1187 train_acc=0.9395 val_loss=0.1956 val_acc=0.9279


[A5_cbam_phase2] epoch 21/40 train_loss=0.1122 train_acc=0.9424 val_loss=0.2026 val_acc=0.9307


[A5_cbam_phase2] epoch 22/40 train_loss=0.1101 train_acc=0.9453 val_loss=0.2218 val_acc=0.9181


[A5_cbam_phase2] epoch 23/40 train_loss=0.1104 train_acc=0.9444 val_loss=0.1951 val_acc=0.9329
[A5_cbam_phase2] Early stopping at epoch 23 (no improvement for 5 epochs).



=== Test set performance (finetuned) ===
Test accuracy: 0.9291
                 precision    recall  f1-score   support

          COVID     0.9547    0.9723    0.9634       542
   Lung_Opacity     0.8951    0.8991    0.8971       902
         Normal     0.9390    0.9261    0.9325      1529
Viral Pneumonia     0.9378    0.9703    0.9538       202

       accuracy                         0.9291      3175
      macro avg     0.9317    0.9420    0.9367      3175
   weighted avg     0.9291    0.9291    0.9291      3175

Confusion matrix:
 [[ 527    6    6    3]
 [  10  811   81    0]
 [  14   89 1416   10]
 [   1    0    5  196]]
Macro ROC-AUC: 0.9899

=== Six-arm summary ===
                  accuracy  macro_f1
A4_guidance_only  0.918740  0.921572
A5_cbam           0.929134  0.936708


## λ_bg Sweep — RUN THIS ONLY AFTER A4 AND A5 FINISH

Independent sweep, `gate_mode="residual"`, `λ_att` fixed at 0.3 (the §8a value). Sweep λ_bg over
{0, 0.1, 0.3, 0.5, 1.0}, short schedule, picking the largest λ_bg whose val macro-F1 stays within
0.5pp of the λ_bg=0 (plain A2) reference; ties broken by higher ILAR (fraction of attention mass
inside the lung mask on validation).

In [10]:
def attention_guidance_loss(att_logits: torch.Tensor, masks: torch.Tensor, target_mode: str = "soft") -> torch.Tensor:
    if masks.ndim == 3:
        masks = masks.unsqueeze(1)
    masks = masks.float()
    h, w = att_logits.shape[-2:]
    target = F.adaptive_avg_pool2d(masks, (h, w)).clamp(0.0, 1.0)
    if target_mode == "hard":
        target = (target > 0.5).float()
    return F.binary_cross_entropy_with_logits(att_logits, target)

def background_suppression_loss(att: torch.Tensor, masks: torch.Tensor) -> torch.Tensor:
    if masks.ndim == 3:
        masks = masks.unsqueeze(1)
    masks = masks.float()
    h, w = att.shape[-2:]
    target = F.adaptive_avg_pool2d(masks, (h, w)).clamp(0.0, 1.0)
    return (att * (1.0 - target)).mean()

def compute_total_loss(class_logits, att_logits, labels, masks, classification_criterion,
                        lambda_att: float = 0.5, att: torch.Tensor = None, lambda_bg: float = 0.0):
    cls_loss = classification_criterion(class_logits, labels)
    total = cls_loss
    if att_logits is not None and lambda_att != 0.0:
        att_loss = attention_guidance_loss(att_logits, masks)
        total = total + lambda_att * att_loss
    else:
        att_loss = torch.zeros_like(cls_loss)
    if att is not None and lambda_bg != 0.0:
        bg_loss = background_suppression_loss(att, masks)
        total = total + lambda_bg * bg_loss
    else:
        bg_loss = torch.zeros_like(cls_loss)
    return total, cls_loss, att_loss, bg_loss

def run_epoch(model, loader, criterion, optimizer, device, train: bool, desc: str = "",
              lambda_att: float = 0.5, lambda_bg: float = 0.0):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    total_cls_loss, total_att_loss, total_bg_loss = 0.0, 0.0, 0.0
    non_blocking = device.type == "cuda"
    context = torch.enable_grad() if train else torch.no_grad()
    progress = tqdm(loader, desc=desc, leave=False, dynamic_ncols=True)
    with context:
        for images, labels, masks in progress:
            images = images.to(device, non_blocking=non_blocking)
            labels = labels.to(device, non_blocking=non_blocking)
            masks = masks.to(device, non_blocking=non_blocking)
            if train:
                optimizer.zero_grad()
            outputs, att, att_logits = model(images)
            loss, cls_loss, att_loss, bg_loss = compute_total_loss(
                outputs, att_logits, labels, masks, criterion,
                lambda_att=lambda_att, att=att, lambda_bg=lambda_bg,
            )
            if train:
                loss.backward()
                optimizer.step()
            batch_size = images.size(0)
            total_loss += loss.item() * batch_size
            total_cls_loss += cls_loss.item() * batch_size
            total_att_loss += att_loss.item() * batch_size
            total_bg_loss += bg_loss.item() * batch_size
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += batch_size
            progress.set_postfix(
                loss=f"{total_loss / total:.4f}", acc=f"{correct / total:.4f}",
                att_loss=f"{total_att_loss / total:.4f}", bg_loss=f"{total_bg_loss / total:.4f}",
            )
    return total_loss / total, correct / total

def train_phase(model, train_loader, val_loader, criterion, optimizer, scheduler,
                 device, epochs, patience, phase_name, output_dir,
                 lambda_att: float = 0.5, lambda_bg: float = 0.0):
    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0
    history = []
    for epoch in range(1, epochs + 1):
        train_desc = f"{phase_name} epoch {epoch}/{epochs} train"
        val_desc = f"{phase_name} epoch {epoch}/{epochs} val"
        train_loss, train_acc = run_epoch(
            model, train_loader, criterion, optimizer, device, train=True,
            desc=train_desc, lambda_att=lambda_att, lambda_bg=lambda_bg,
        )
        val_loss, val_acc = run_epoch(
            model, val_loader, criterion, optimizer, device, train=False,
            desc=val_desc, lambda_att=lambda_att, lambda_bg=lambda_bg,
        )
        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_loss)
            else:
                scheduler.step()
        history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
                         "val_loss": val_loss, "val_acc": val_acc})
        print(f"[{phase_name}] epoch {epoch}/{epochs} "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"[{phase_name}] Early stopping at epoch {epoch} (no improvement for {patience} epochs).")
                break
    model.load_state_dict(best_state)
    with open(Path(output_dir) / f"{phase_name}_history.json", "w") as f:
        json.dump(history, f, indent=2)
    return model

@torch.no_grad()
def evaluate(model, loader, class_names, device, output_dir, evaluation_type):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    non_blocking = device.type == "cuda"
    for images, labels, masks in loader:
        images = images.to(device, non_blocking=non_blocking)
        outputs, att, att_logits = model(images)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)
        all_labels.extend(labels.numpy())
        all_preds.extend(preds)
        all_probs.extend(probs)
    all_labels = np.array(all_labels); all_preds = np.array(all_preds); all_probs = np.array(all_probs)
    report = classification_report(all_labels, all_preds, target_names=class_names, digits=4, output_dict=True)
    cm = confusion_matrix(all_labels, all_preds)
    try:
        auc_macro = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro")
        auc_per_class = roc_auc_score(all_labels, all_probs, multi_class="ovr", average=None)
    except ValueError:
        auc_macro, auc_per_class = None, None
    results = {"classification_report": report, "confusion_matrix": cm.tolist(),
               "roc_auc_macro": auc_macro,
               "roc_auc_per_class": auc_per_class.tolist() if auc_per_class is not None else None,
               "class_names": class_names}
    with open(Path(output_dir) / f"{evaluation_type}_test_results.json", "w") as f:
        json.dump(results, f, indent=2)
    print(f"\n=== Test set performance ({evaluation_type}) ===")
    print(f"Test accuracy: {accuracy_score(all_labels, all_preds):.4f}")
    print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))
    print("Confusion matrix:\n", cm)
    if auc_macro is not None:
        print(f"Macro ROC-AUC: {auc_macro:.4f}")
    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
    cm_df.to_csv(Path(output_dir) / f"{evaluation_type}_confusion_matrix.csv")
    summary = {"accuracy": accuracy_score(all_labels, all_preds),
               "macro_f1": report["macro avg"]["f1-score"],
               "macro_recall": report["macro avg"]["recall"],
               "macro_precision": report["macro avg"]["precision"],
               "weighted_f1": report["weighted avg"]["f1-score"],
               "weighted_recall": report["weighted avg"]["recall"],
               "roc_auc_macro": auc_macro}
    pd.DataFrame([summary]).to_csv(Path(output_dir) / f"{evaluation_type}_summary_metrics.csv", index=False)
    return results

def build_optimizer(model, name, lr, weight_decay):
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    if name == "adamw": return torch.optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)
    elif name == "adam": return torch.optim.Adam(trainable_params, lr=lr, weight_decay=weight_decay)
    elif name == "sgd": return torch.optim.SGD(trainable_params, lr=lr, momentum=0.9, weight_decay=weight_decay)
    raise ValueError(f"Unknown optimizer: {name}")

def build_scheduler(optimizer, name, epochs):
    if name == "cosine": return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    elif name == "step": return torch.optim.lr_scheduler.StepLR(optimizer, step_size=max(1, epochs // 3), gamma=0.1)
    elif name == "plateau": return torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.1, patience=2)
    elif name == "none": return None
    raise ValueError(f"Unknown scheduler: {name}")

def freeze_vit_backbone(model):
    for param in model.parameters(): param.requires_grad = False
    for p in model.backbone.get_classifier().parameters(): p.requires_grad = True
    if model.attn is not None:
        for p in model.attn.parameters(): p.requires_grad = True

def unfreeze_vit_blocks(model, num_blocks_to_unfreeze: int = 2):
    for param in model.parameters(): param.requires_grad = False
    for p in model.backbone.get_classifier().parameters(): p.requires_grad = True
    if model.attn is not None:
        for p in model.attn.parameters(): p.requires_grad = True
    for name, param in model.backbone.named_parameters():
        if "norm" in name and "blocks" not in name:
            param.requires_grad = True
    total_blocks = len(model.backbone.blocks)
    target_blocks = [f"blocks.{total_blocks - 1 - i}" for i in range(num_blocks_to_unfreeze)]
    for name, param in model.backbone.named_parameters():
        if any(b in name for b in target_blocks):
            param.requires_grad = True

## Train A6 (full+bg)

In [11]:
from sklearn.metrics import classification_report

LAMBDA_STAR = 0.3

sweep_lambda_bg_values = [0.0, 0.1, 0.3, 0.5, 1.0]
sweep_epochs = 3
bg_sweep_results = []

@torch.no_grad()
def mean_ilar(model, loader, device, n_batches=10):
    """Fraction of attention mass falling inside the lung mask, averaged over a few val batches."""
    model.eval()
    ratios = []
    for i, (images, labels, masks) in enumerate(loader):
        if i >= n_batches:
            break
        images, masks = images.to(device), masks.to(device)
        _, att, _ = model(images)
        if att is None:
            continue
        if masks.ndim == 3:
            masks = masks.unsqueeze(1)
        h, w = att.shape[-2:]
        target = F.adaptive_avg_pool2d(masks.float(), (h, w)).clamp(0.0, 1.0)
        inside = (att * target).sum(dim=[1, 2, 3])
        total = att.sum(dim=[1, 2, 3]).clamp(min=1e-8)
        ratios.extend((inside / total).cpu().numpy())
    return float(np.mean(ratios)) if ratios else float("nan")

@torch.no_grad()
def val_macro_f1(model, loader, device, class_names):
    """Macro-F1 on the validation set -- the metric the winner-selection rule is actually based on."""
    model.eval()
    all_labels, all_preds = [], []
    for images, labels, masks in loader:
        images = images.to(device)
        outputs, _, _ = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_labels.extend(labels.numpy())
        all_preds.extend(preds)
    report = classification_report(
        all_labels, all_preds, target_names=class_names, output_dict=True, zero_division=0
    )
    return report["macro avg"]["f1-score"]

for lam_bg in sweep_lambda_bg_values:
    print(f"\n=== BG sweep trial: lambda_bg={lam_bg} ===")
    set_seed(args.seed)

    bg_model = ViTLungAttention(
        num_classes=num_classes, use_attention=True, attention="lung", gate_mode="residual"
    ).to(device)
    freeze_vit_backbone(bg_model)
    bg_opt = build_optimizer(bg_model, args.optimizer, args.phase1_lr, args.weight_decay)
    bg_sched = build_scheduler(bg_opt, args.scheduler, sweep_epochs)

    bg_out_dir = f"./runs/bg_sweep/lambda_bg_{lam_bg}"
    Path(bg_out_dir).mkdir(parents=True, exist_ok=True)

    bg_model = train_phase(
        bg_model, train_loader, val_loader, criterion, bg_opt, bg_sched,
        device, sweep_epochs, patience=sweep_epochs, phase_name=f"bgsweep_{lam_bg}",
        output_dir=bg_out_dir, lambda_att=LAMBDA_STAR, lambda_bg=lam_bg,
    )

    val_loss, val_acc = run_epoch(
        bg_model, val_loader, criterion, None, device, train=False,
        desc=f"final_val_bg_{lam_bg}", lambda_att=LAMBDA_STAR, lambda_bg=lam_bg,
    )
    macro_f1 = val_macro_f1(bg_model, val_loader, device, class_names)
    ilar = mean_ilar(bg_model, val_loader, device)

    bg_sweep_results.append({
        "lambda_bg": lam_bg,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_macro_f1": macro_f1,
        "ilar": ilar,
    })
    print(f"  -> val_acc={val_acc:.4f}  val_macro_f1={macro_f1:.4f}  ilar={ilar:.4f}")

bg_sweep_df = pd.DataFrame(bg_sweep_results)
print("\n=== BG sweep summary ===")
print(bg_sweep_df)

Path("./runs/bg_sweep").mkdir(parents=True, exist_ok=True)
bg_sweep_df.to_csv("./runs/bg_sweep/bg_sweep_summary.csv", index=False)


=== BG sweep trial: lambda_bg=0.0 ===


[bgsweep_0.0] epoch 1/3 train_loss=0.7493 train_acc=0.7114 val_loss=0.5745 val_acc=0.7745


[bgsweep_0.0] epoch 2/3 train_loss=0.5302 train_acc=0.7949 val_loss=0.4976 val_acc=0.8198


[bgsweep_0.0] epoch 3/3 train_loss=0.4835 train_acc=0.8202 val_loss=0.4855 val_acc=0.8107


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


...

[bgsweep_1.0] epoch 1/3 train_loss=0.8043 train_acc=0.7107 val_loss=0.6112 val_acc=0.7729


[bgsweep_1.0] epoch 2/3 train_loss=0.5665 train_acc=0.7943 val_loss=0.5319 val_acc=0.8189


[bgsweep_1.0] epoch 3/3 train_loss=0.5186 train_acc=0.8198 val_loss=0.5193 val_acc=0.8117


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


  -> val_acc=0.8117  val_macro_f1=0.7938  ilar=0.8775

=== BG sweep summary ===
   lambda_bg  val_loss   val_acc  val_macro_f1      ilar
0        0.0  0.485500  0.810709      0.792501  0.788259
1        0.1  0.489632  0.810709      0.792501  0.807394
2        0.3  0.497805  0.810709      0.792678  0.831872
3        0.5  0.504885  0.810394      0.792348  0.850612
4        1.0  0.519344  0.811654      0.793754  0.877525


## train A6 at seed 42

In [16]:
LAMBDA_ATT_STAR = 0.3
LAMBDA_BG_STAR = 1.0  # confirm this matches what your selection cell printed

set_seed(args.seed)  # re-seed before training, same convention as every other arm

a6_model = ViTLungAttention(
    num_classes=num_classes, use_attention=True, attention="lung", gate_mode="residual"
).to(device)

a6_out_dir = "./runs/six_arm/A6_full_bg"
Path(a6_out_dir).mkdir(parents=True, exist_ok=True)
print(f"Output directory for A6: {a6_out_dir}")

# --- Phase 1: frozen backbone ---
print(f"\n{'='*20} ARM: A6_full_bg -- Phase 1 {'='*20}")
freeze_vit_backbone(a6_model)
opt1 = build_optimizer(a6_model, args.optimizer, args.phase1_lr, args.weight_decay)
sched1 = build_scheduler(opt1, args.scheduler, args.phase1_epochs)

a6_model = train_phase(
    a6_model, train_loader, val_loader, criterion, opt1, sched1,
    device, args.phase1_epochs, args.patience, "A6_full_bg_phase1", a6_out_dir,
    lambda_att=LAMBDA_ATT_STAR, lambda_bg=LAMBDA_BG_STAR,
)
evaluate(a6_model, test_loader, class_names, device, a6_out_dir, "frozen")

# --- Phase 2: unfreeze last N blocks ---
print(f"\n{'='*20} ARM: A6_full_bg -- Phase 2 {'='*20}")
unfreeze_vit_blocks(a6_model, num_blocks_to_unfreeze=args.unfreeze_blocks)
opt2 = build_optimizer(a6_model, args.optimizer, args.phase2_lr, args.weight_decay)
sched2 = build_scheduler(opt2, args.scheduler, args.phase2_epochs)

a6_model = train_phase(
    a6_model, train_loader, val_loader, criterion, opt2, sched2,
    device, args.phase2_epochs, args.patience, "A6_full_bg_phase2", a6_out_dir,
    lambda_att=LAMBDA_ATT_STAR, lambda_bg=LAMBDA_BG_STAR,
)

print("\n=== A6 test set performance (finetuned) ===")
a6_result = evaluate(a6_model, test_loader, class_names, device, a6_out_dir, "finetuned")

arm_results_A6 = {
    "accuracy": a6_result["classification_report"]["accuracy"],
    "macro_f1": a6_result["classification_report"]["macro avg"]["f1-score"],
    "roc_auc_macro": a6_result["roc_auc_macro"],
    "lambda_att": LAMBDA_ATT_STAR,
    "lambda_bg": LAMBDA_BG_STAR,
}
print(f"\nA6 summary: {arm_results_A6}")

with open(Path(a6_out_dir) / "A6_arm_summary.json", "w") as f:
    json.dump(arm_results_A6, f, indent=2)

Output directory for A6: ./runs/six_arm/A6_full_bg

==================== ARM: A6_full_bg -- Phase 1 ====================


[A6_full_bg_phase1] epoch 1/15 train_loss=0.8043 train_acc=0.7107 val_loss=0.6112 val_acc=0.7729


[A6_full_bg_phase1] epoch 2/15 train_loss=0.5618 train_acc=0.7939 val_loss=0.5263 val_acc=0.8135


[A6_full_bg_phase1] epoch 3/15 train_loss=0.5045 train_acc=0.8215 val_loss=0.4847 val_acc=0.8306


[A6_full_bg_phase1] epoch 4/15 train_loss=0.4712 train_acc=0.8308 val_loss=0.4529 val_acc=0.8362


...

[A6_full_bg_phase2] epoch 16/40 train_loss=0.2116 train_acc=0.9324 val_loss=0.3014 val_acc=0.9238


[A6_full_bg_phase2] epoch 17/40 train_loss=0.2051 train_acc=0.9342 val_loss=0.2793 val_acc=0.9231


[A6_full_bg_phase2] epoch 18/40 train_loss=0.2069 train_acc=0.9370 val_loss=0.2607 val_acc=0.9263


[A6_full_bg_phase2] epoch 19/40 train_loss=0.2009 train_acc=0.9370 val_loss=0.2804 val_acc=0.9260
[A6_full_bg_phase2] Early stopping at epoch 19 (no improvement for 5 epochs).

=== A6 test set performance (finetuned) ===



=== Test set performance (finetuned) ===
Test accuracy: 0.9178
                 precision    recall  f1-score   support

          COVID     0.9402    0.9576    0.9488       542
   Lung_Opacity     0.9141    0.8614    0.8870       902
         Normal     0.9197    0.9287    0.9242      1529
Viral Pneumonia     0.8646    0.9802    0.9188       202

       accuracy                         0.9178      3175
      macro avg     0.9097    0.9320    0.9197      3175
   weighted avg     0.9181    0.9178    0.9175      3175

Confusion matrix:
 [[ 519    9   10    4]
 [  14  777  110    1]
 [  19   64 1420   26]
 [   0    0    4  198]]
Macro ROC-AUC: 0.9880

A6 summary: {'accuracy': 0.9177952755905512, 'macro_f1': 0.9196924581000876, 'roc_auc_macro': np.float64(0.9880148312041794), 'lambda_att': 0.3, 'lambda_bg': 1.0}


### Six-arm ablation result — A0 through A6

**λ_att\* = 0.3** (from λ_att sweep), **λ_bg\* = 1.0** (from λ_bg sweep, selected via §8b rule:
largest λ_bg whose val macro-F1 stays within 0.5pp of λ_bg=0, tie-broken by ILAR).

| Rank | Arm | Test Acc | Macro-F1 | Macro-AUC |
|---|---|---|---|---|
| 1 | **A5 CBAM** | 0.9291 | **0.9367** | 0.9899 |
| 2 | A3 multiply-gate | 0.9203 | 0.9264 | 0.9877 |
| 3 | A1 gate-only | 0.9209 | 0.9242 | 0.9883 |
| 4 | A0 vanilla | 0.9181 | 0.9241 | 0.9883 |
| 5 | A2 full | 0.9184 | 0.9208 | 0.9880 |
| 6 | A4 guidance-only | 0.9187 | 0.9216 | 0.9879 |
| 7 | A6 full+bg | 0.9178 | 0.9197 | 0.9880 |

**A5 (CBAM) leads on all three classification metrics**, ahead of the next-best arm (A3) by
+0.88pp accuracy and +1.03pp macro-F1.

**A6 (full attention + background suppression) lands at the bottom** on classification
metrics — on par with A0 vanilla, ~0.6pp macro-F1 behind A2 (its own architecture without
bg suppression). λ_bg=1.0 pushed validation ILAR from 0.788 (λ_bg=0) to 0.878, so background
suppression did concentrate attention inside the lung mask as intended — it just came with a
small classification cost on this run.

**Caveat:** this table is classification metrics only. Per the guide's winner-selection rule
(§11), the real decision requires attention-Dice, Grad-CAM EIL, and McNemar/Wilcoxon
significance tests vs. A0 — not run yet. A5 leading here does not guarantee it wins once
attention-quality metrics and significance testing are applied.